In [1]:
import pandas as pd
import numpy as np
import sqlite3
from great_tables import gt

In [2]:
conn=sqlite3.connect('Data/wnba_database.db')
current_wnba_season=2026

# Read CSV
pbp_curr_yr = pd.read_csv(f"Data/{current_wnba_season}_pbp.csv")

# Group by gameId and fill scores within each game
pbp_curr_yr[['scoreHome', 'scoreAway']] = (
    pbp_curr_yr.groupby('gameId')[['scoreHome', 'scoreAway']]
    .ffill()
)

# Score margin
pbp_curr_yr['scoreMargin'] = pbp_curr_yr['scoreHome'] - pbp_curr_yr['scoreAway']

# Remove All-Star weekend (gameId starting with '3')
pbp_curr_yr = pbp_curr_yr[
    ~pbp_curr_yr['gameId'].astype(str).str.startswith('3')
]

# Reset index
pbp_curr_yr = pbp_curr_yr.reset_index(drop=True)

In [3]:
# --- compact_standings ---
standings=(
    pd.read_sql_query(
        'SELECT * FROM standings WHERE season = ?',
        con=conn,
        params=[current_wnba_season])
)

# --- player_bio ---
player_bio=(
    pd.read_sql_query(
        'SELECT * FROM player_bios WHERE season = ?',
        con=conn,
        params=[current_wnba_season])
)

# --- team_info ---
team_info = (
    standings
    .merge(
        player_bio
        .sort_values("TEAM_ID")
        .groupby("TEAM_ID", as_index=False)
        .first()[["TEAM_ID","TEAM_ABBREVIATION"]],
        left_on="TeamID",
        right_on="TEAM_ID",
        how="left"
    )
)

player_totals_w_gp_percentages=(
    player_bio
    .merge(
        standings.filter(regex="Team"),
        left_on="TEAM_ID",
        right_on="TeamID",
        how="left"
    )
    .merge(
        pd.read_sql_query(
            'SELECT * FROM player_totals WHERE season = ?',
            con=conn,
            params=[current_wnba_season]
            ).loc[:, ["PLAYER_ID","MIN"]],
        on="PLAYER_ID",
        how="left"
    )
    .assign(G_PERCENT=lambda d: d.GP / d.TeamGP)
)
del standings, player_bio

# Awards

## The "Chicken Supplier" Award (sponsored by Los Pollos Hermanos)

most pairs of free throws missed by a visiting player in the second half (credit to livejamie for the idea)

In [4]:
missed_visitor_ft_in_second_half = (
    pbp_curr_yr
    .loc[
        (pbp_curr_yr["actionType"] == "Free Throw") &
        (pbp_curr_yr["description"].str.startswith("MISS", na=False)) &
        (pbp_curr_yr["location"] == "v") &
        (pbp_curr_yr["period"] > 2) &
        (pbp_curr_yr["subType"].str.endswith(("2", "3"), na=False))
    ]
    .groupby(["personId", "playerNameI", "gameId", "clock", "period"])
    .size()
    .reset_index(name="n")
    .query("n > 1")
    .groupby(["personId", "playerNameI"], as_index=False)
    .agg(missed_ft_pairs=("n", "size"))
    .merge(
        player_totals_w_gp_percentages[["PLAYER_ID", "PLAYER_NAME"]]
        .rename(columns={"PLAYER_ID": "personId","PLAYER_NAME": "player_name"}),
        on="personId",
        how="left"
    )
)

In [5]:
gt.GT(
    missed_visitor_ft_in_second_half
    .drop(columns=["playerNameI", "personId"])
    .nlargest(5, "missed_ft_pairs",keep="all")
)

missed_ft_pairs,player_name
4,Angel Reese
3,Jessica Shepard
3,Shakira Austin
3,Aneesah Morrow
2,Tiffany Hayes
2,Alyssa Thomas
2,Natasha Howard
2,Elizabeth Williams
2,Natasha Cloud
2,Jonquel Jones


## The "Rent-Free" Award (presented by Monica Geller)

team that has the highest differential of technical & flagrant FTAs taken vs given up (credit to whackedjob for the idea & Drummallumin for the original idea of highest total attempts taken)

In [6]:
team_and_opponent_by_game=(
        pbp_curr_yr[["gameId", "teamTricode"]]
            .drop_duplicates()
            .query("teamTricode.notna()")
            .merge(
                pbp_curr_yr[["gameId", "teamTricode"]]
                .drop_duplicates()
                .query("teamTricode.notna()"),
                on="gameId",
                suffixes=("", "_opp"),
            )
            .query("teamTricode != teamTricode_opp")
            .rename(columns={"teamTricode_opp": "opponent"})[
                ["gameId", "teamTricode", "opponent"]
            ]
            .drop_duplicates()
)

flagrant_tech_ftas=(
    pbp_curr_yr.loc[
        pbp_curr_yr["subType"].str.contains(
            "Free Throw Technical|Free Throw Flagrant", na=False
        )
    ].merge(team_and_opponent_by_game,
        how='left'
    )
)

In [7]:
flagrant_tech_fta_received=(
    flagrant_tech_ftas.groupby("teamTricode", as_index=False)
    .agg(
        tech_fta_received=(
            "subType",
            lambda s: s.str.contains("Free Throw Technical", na=False).sum(),
        ),
        flagrant_fta_received=(
            "subType",
            lambda s: s.str.contains("Free Throw Flagrant", na=False).sum(),
        ),
    )
    .assign(
        flagrant_plus_tech_fta_received=lambda d: (
            d.tech_fta_received + d.flagrant_fta_received
        )
    )
)

flagrant_tech_fta_given=(
    flagrant_tech_ftas.groupby("opponent", as_index=False)
    .agg(
        tech_fta_given=(
            "subType",
            lambda s: s.str.contains("Free Throw Technical", na=False).sum(),
        ),
        flagrant_fta_given=(
            "subType",
            lambda s: s.str.contains("Free Throw Flagrant", na=False).sum(),
        ),
    )
    .assign(
        flagrant_plus_tech_fta_given=lambda d: d.tech_fta_given + d.flagrant_fta_given
    )
    )

flagrant_tech_team_summ=flagrant_tech_fta_received.merge(flagrant_tech_fta_given,
    how="left",
    left_on="teamTricode",
    right_on="opponent",
).assign(
    flagrant_plus_tech_differential=lambda d: (
        d.flagrant_plus_tech_fta_received - d.flagrant_plus_tech_fta_given
    )
).drop(axis=1,labels=['opponent'])

del flagrant_tech_fta_given,flagrant_tech_fta_received

In [8]:
gt.GT(
    flagrant_tech_team_summ.nlargest(5, "flagrant_plus_tech_differential", keep="all")
).tab_spanner(
    label="Received",
    columns=[
        "tech_fta_received",
        "flagrant_fta_received",
        "flagrant_plus_tech_fta_received",
    ],
).tab_spanner(
    label="Given",
    columns=[
        "tech_fta_given",
        "flagrant_fta_given",
        "flagrant_plus_tech_fta_given",
    ],
).cols_label(
    teamTricode='Team',
    tech_fta_received="Tech",tech_fta_given="Tech",
    flagrant_fta_received="Flagrant",flagrant_fta_given="Flagrant",
    flagrant_plus_tech_fta_received="Total",
    flagrant_plus_tech_fta_given="Total",
    flagrant_plus_tech_differential="Diff"
)

GT(_tbl_data=   teamTricode  tech_fta_received  flagrant_fta_received  ...  flagrant_fta_given  flagrant_plus_tech_fta_given  flagrant_plus_tech_differential
4          GSV                 18                     15  ...                   2                            17                               16
13         TOR                 19                     13  ...                   7                            19                               13
0          ATL                 27                     16  ...                  13                            36                                7
11         PHX                 17                     11  ...                   4                            22                                6
1          CHI                 18                     12  ...                   4                            26                                4

[5 rows x 8 columns], _body=<great_tables._gt_data.Body object at 0x16c3b25d0>, _boxhead=Boxhead([ColInfo(var='teamTricode', type=<ColInfoTypeEnum.default: 1>, column_label='Team', column_align='left', column_width=None), ColInfo(var='tech_fta_received', type=<ColInfoTypeEnum.default: 1>, column_label='Tech', column_align='right', column_width=None), ColInfo(var='flagrant_fta_received', type=<ColInfoTypeEnum.default: 1>, column_label='Flagrant', column_align='right', column_width=None), ColInfo(var='flagrant_plus_tech_fta_received', type=<ColInfoTypeEnum.default: 1>, column_label='Total', column_align='right', column_width=None), ColInfo(var='tech_fta_given', type=<ColInfoTypeEnum.default: 1>, column_label='Tech', column_align='right', column_width=None), ColInfo(var='flagrant_fta_given', type=<ColInfoTypeEnum.default: 1>, column_label='Flagrant', column_align='right', column_width=None), ColInfo(var='flagrant_plus_tech_fta_given', type=<ColInfoTypeEnum.default: 1>, column_label='Total', column_align='right', column_width=None), ColInfo(var='flagrant_plus_tech_differential', type=<ColInfoTypeEnum.default: 1>, column_label='Diff', column_align='right', column_width=None)]), _stub=<great_tables._gt_data.Stub object at 0x1789c0c50>, _spanners=Spanners([SpannerInfo(spanner_id='Received', spanner_level=0, spanner_label='Received', spanner_units=None, spanner_pattern=None, vars=['tech_fta_received', 'flagrant_fta_received', 'flagrant_plus_tech_fta_received'], built=None), SpannerInfo(spanner_id='Given', spanner_level=0, spanner_label='Given', spanner_units=None, spanner_pattern=None, vars=['tech_fta_given', 'flagrant_fta_given', 'flagrant_plus_tech_fta_given'], built=None)]), _heading=Heading(title=None, subtitle=None, preheader=None), _stubhead=None, _summary_rows=<great_tables._gt_data.SummaryRows object at 0x1789b2390>, _summary_rows_grand=<great_tables._gt_data.SummaryRows object at 0x1789b2fd0>, _source_notes=[], _footnotes=[], _styles=[], _locale=<great_tables._gt_data.Locale object at 0x1789b19d0>, _formats=[], _substitutions=[], _col_merge=[], _options=Options(table_id=OptionsInfo(scss=False, category='table', type='value', value=None), table_caption=OptionsInfo(scss=False, category='table', type='value', value=None), table_width=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_layout=OptionsInfo(scss=True, category='table', type='value', value='fixed'), table_margin_left=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_margin_right=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_background_color=OptionsInfo(scss=True, category='table', type='value', value='#FFFFFF'), table_additional_css=OptionsInfo(scss=False, category='table', type='values', value=[]), table_font_names=OptionsInfo(scss=False, category='table', type='values', value=['-apple-system', 'BlinkMacSystemFont', 'Segoe UI', 'Roboto', 'Oxygen', 'Ubuntu', 'Cantarell', 'Helvetica Neue', 'Fira Sans', 'Droid Sans', 'Arial', 'sans-serif']), table_font_size=OptionsInfo(scss=True, category='table', type='px', value='16px'), table_fon

## The "Anger Management Classes Start on Tuesday" Award (presented by Lewis Black)

team that has the lowest differential of technical & flagrant FTAs taken vs given up

In [9]:
gt.GT(
    flagrant_tech_team_summ.nsmallest(5, "flagrant_plus_tech_differential", keep="all")
).tab_spanner(
    label="Received",
    columns=[
        "tech_fta_received",
        "flagrant_fta_received",
        "flagrant_plus_tech_fta_received",
    ],
).tab_spanner(
    label="Given",
    columns=[
        "tech_fta_given",
        "flagrant_fta_given",
        "flagrant_plus_tech_fta_given",
    ],
).cols_label(
    teamTricode='Team',
    tech_fta_received="Tech",tech_fta_given="Tech",
    flagrant_fta_received="Flagrant",flagrant_fta_given="Flagrant",
    flagrant_plus_tech_fta_received="Total",
    flagrant_plus_tech_fta_given="Total",
    flagrant_plus_tech_differential="Diff"
)

GT(_tbl_data=   teamTricode  tech_fta_received  flagrant_fta_received  ...  flagrant_fta_given  flagrant_plus_tech_fta_given  flagrant_plus_tech_differential
9          NYL                 14                      2  ...                  11                            36                              -20
8          MIN                 13                      9  ...                  15                            32                              -10
6          LAS                 19                      5  ...                  15                            33                               -9
5          IND                 15                     10  ...                   6                            32                               -7
14         WAS                 13                      4  ...                  10                            22                               -5

[5 rows x 8 columns], _body=<great_tables._gt_data.Body object at 0x175d37690>, _boxhead=Boxhead([ColInfo(var='teamTricode', type=<ColInfoTypeEnum.default: 1>, column_label='Team', column_align='left', column_width=None), ColInfo(var='tech_fta_received', type=<ColInfoTypeEnum.default: 1>, column_label='Tech', column_align='right', column_width=None), ColInfo(var='flagrant_fta_received', type=<ColInfoTypeEnum.default: 1>, column_label='Flagrant', column_align='right', column_width=None), ColInfo(var='flagrant_plus_tech_fta_received', type=<ColInfoTypeEnum.default: 1>, column_label='Total', column_align='right', column_width=None), ColInfo(var='tech_fta_given', type=<ColInfoTypeEnum.default: 1>, column_label='Tech', column_align='right', column_width=None), ColInfo(var='flagrant_fta_given', type=<ColInfoTypeEnum.default: 1>, column_label='Flagrant', column_align='right', column_width=None), ColInfo(var='flagrant_plus_tech_fta_given', type=<ColInfoTypeEnum.default: 1>, column_label='Total', column_align='right', column_width=None), ColInfo(var='flagrant_plus_tech_differential', type=<ColInfoTypeEnum.default: 1>, column_label='Diff', column_align='right', column_width=None)]), _stub=<great_tables._gt_data.Stub object at 0x16c3b1390>, _spanners=Spanners([SpannerInfo(spanner_id='Received', spanner_level=0, spanner_label='Received', spanner_units=None, spanner_pattern=None, vars=['tech_fta_received', 'flagrant_fta_received', 'flagrant_plus_tech_fta_received'], built=None), SpannerInfo(spanner_id='Given', spanner_level=0, spanner_label='Given', spanner_units=None, spanner_pattern=None, vars=['tech_fta_given', 'flagrant_fta_given', 'flagrant_plus_tech_fta_given'], built=None)]), _heading=Heading(title=None, subtitle=None, preheader=None), _stubhead=None, _summary_rows=<great_tables._gt_data.SummaryRows object at 0x178979710>, _summary_rows_grand=<great_tables._gt_data.SummaryRows object at 0x178979c90>, _source_notes=[], _footnotes=[], _styles=[], _locale=<great_tables._gt_data.Locale object at 0x17897a3d0>, _formats=[], _substitutions=[], _col_merge=[], _options=Options(table_id=OptionsInfo(scss=False, category='table', type='value', value=None), table_caption=OptionsInfo(scss=False, category='table', type='value', value=None), table_width=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_layout=OptionsInfo(scss=True, category='table', type='value', value='fixed'), table_margin_left=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_margin_right=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_background_color=OptionsInfo(scss=True, category='table', type='value', value='#FFFFFF'), table_additional_css=OptionsInfo(scss=False, category='table', type='values', value=[]), table_font_names=OptionsInfo(scss=False, category='table', type='values', value=['-apple-system', 'BlinkMacSystemFont', 'Segoe UI', 'Roboto', 'Oxygen', 'Ubuntu', 'Cantarell', 'Helvetica Neue', 'Fira Sans', 'Droid Sans', 'Arial', 'sans-serif']), table_font_size=OptionsInfo(scss=True, category='table', type='px', value='16px'), table_fon

## The "It Ain't Over Till the Fat Lady Sings" Award (presented by Kim Kardashian)

most comeback wins where the opposing team had a lead of at least 10 points at some point in the game

In [10]:
largest_leads = (
    pbp_curr_yr
    .loc[
        pbp_curr_yr["teamTricode"].notna() &
        pbp_curr_yr["location"].isin(["h", "v"])
    ]
    .groupby("gameId")
    .agg(
        vis_final=("scoreAway", "max"),
        home_final=("scoreHome", "max"),
        largest_home_lead=("scoreMargin", "max"),
        largest_vis_lead=("scoreMargin", lambda s: -s.min()),
        home=("teamTricode", lambda s: s[pbp_curr_yr.loc[s.index, "location"] == "h"].iloc[0]),
        visitor=("teamTricode", lambda s: s[pbp_curr_yr.loc[s.index, "location"] == "v"].iloc[0]),
    )
    .reset_index()
    .assign(
        winner_team=lambda d: d["home"].where(
            d["home_final"] > d["vis_final"], d["visitor"]
        ),
        loser_team=lambda d: d["visitor"].where(
            d["home_final"] > d["vis_final"], d["home"]
        ),
        largest_winner_lead=lambda d: d["largest_home_lead"].where(
            d["home_final"] > d["vis_final"], d["largest_vis_lead"]
        ),
        largest_loser_lead=lambda d: d["largest_vis_lead"].where(
            d["home_final"] > d["vis_final"], d["largest_home_lead"]
        )
    )
)

In [11]:
gt.GT(
    largest_leads
    .groupby("winner_team", as_index=False)
    .agg(num_wins_trail_by_10=("largest_loser_lead", lambda s: (s >= 10).sum()))
    .nlargest(5, "num_wins_trail_by_10",keep="all")
)

winner_team,num_wins_trail_by_10
WAS,7
DAL,6
NYL,6
ATL,5
IND,5
LAS,5
PDX,5


## The "Snatching Defeat from the Jaws of Victory" Award (presented by the 28-3 Atlanta Falcons)

most losses when having a lead of at least 10 points at some point in the game

In [12]:
gt.GT(
    largest_leads
    .groupby("loser_team", as_index=False)
    .agg(
        num_losses_lead_by_10=("largest_loser_lead", lambda s: (s >= 10).sum())
    )
    .nlargest(5, "num_losses_lead_by_10",keep="all")
)

loser_team,num_losses_lead_by_10
CON,7
DAL,6
PHX,6
CHI,5
IND,5


## The No Time to Taunt Award*

highest percent of blocks that stayed inbounds & recovered by blocker's team, min 0.7 blocks per game (credit to gibberisle for the idea)

In [13]:
blocks_w_next_play = (
    pbp_curr_yr
    .sort_values(["gameId","actionNumber"])
    .assign(
        next_play=lambda d: d["description"].shift(-1),
        next_location=lambda d: d["location"].shift(-1)
    )
    .loc[lambda d: d["description"].str.contains("BLOCK", na=False)]
    .assign(
        recovered_by=lambda d: (
            (~d["next_play"].str.contains("Rebound", na=False)) &
            (d["location"] == d["next_location"])
        ).map({True: "own", False: "other"})
    )
)

player_blk_info = (
    blocks_w_next_play
    .groupby(["personId", "playerNameI", "recovered_by"], as_index=False)
    .size()
    .rename(columns={"size": "blocks"})
    .pivot(
        index=["personId", "playerNameI"],
        columns="recovered_by",
        values="blocks"
    )
    .fillna(0)
)

player_blk_info.columns = [
    f"blocks_recovered_by_{c}" for c in player_blk_info.columns
]

player_blk_info = (
    player_blk_info
    .reset_index()
    .assign(
        blocks=lambda d: d.blocks_recovered_by_own + d.blocks_recovered_by_other,
        percent_blk_recovered_by_own=lambda d:
        d.blocks_recovered_by_own / d.blocks
    )
    .merge(
        player_totals_w_gp_percentages[["PLAYER_ID", "PLAYER_NAME"]]
        .rename(columns={"PLAYER_ID": "personId","PLAYER_NAME": "player_name"}),
        on="personId",
        how="left"
    )
    .pipe(lambda d: d.assign(player_name=d.pop("player_name")))
    .drop(columns="playerNameI")
)

average_games_played = team_info["TeamGP"].mean()

gt.GT(
        player_blk_info
        .loc[lambda d: d["blocks"] >= average_games_played * 0.7]
        .nlargest(10, "percent_blk_recovered_by_own",keep="all")
        .rename(columns={"blocks_recovered_by_own": "own"})
        [["player_name", "blocks", "own", "percent_blk_recovered_by_own"]]
    ).fmt_percent(columns=["percent_blk_recovered_by_own"])

player_name,blocks,own,percent_blk_recovered_by_own
Dominique Malonga,48.0,31.0,64.58%
Jonquel Jones,50.0,29.0,58.00%
Nia Brodie,42.0,24.0,57.14%
Nia Brodie,42.0,24.0,57.14%
Nia Coffey,42.0,24.0,57.14%
Nia Coffey,42.0,24.0,57.14%
Lauren Betts,37.0,21.0,56.76%
Natasha Mack,48.0,27.0,56.25%
Angel Reese,32.0,18.0,56.25%
Makayla Timpson,34.0,17.0,50.00%


## The All-Ball Award*

most 3-pt shooting fouls committed (credit to watchingsongsDL, kingcobweb & An-Indian-In-The-NBA for the idea, and sunnysideoutside for the name)

In [14]:
fouls_on_threes = (
    pbp_curr_yr
    # shooting fouls
    .loc[pbp_curr_yr["description"].str.contains("S.FOUL", na=False)]
    .merge(
        pbp_curr_yr.loc[
            (
                pbp_curr_yr["description"].str.contains("Free Throw 3 of 3", na=False)
            ) |
            (
                pbp_curr_yr["description"].str.contains("3PT", na=False) &
                (pbp_curr_yr["shotResult"] == "Made")
            )
        ],
        on=["gameId", "period", "clock"],
        how="inner",
        suffixes=(".x", ".y")
    )
)

In [15]:
gt.GT(
    fouls_on_threes
    .groupby(["personId.x", "playerNameI.x"], as_index=False)
    .size()
    .rename(columns={"size": "count"})
    .merge(
        player_totals_w_gp_percentages[["PLAYER_ID", "PLAYER_NAME"]]
        .rename(columns={
            "PLAYER_ID": "personId.x",
            "PLAYER_NAME": "player_name"
        }),
        on="personId.x",
        how="left"
    )
    .pipe(lambda d: d.assign(player_name=d.pop("player_name")))
    .drop(columns=["playerNameI.x", "personId.x"])
    .nlargest(5, "count", keep="all")
)

count,player_name
7,Arike Ogunbowale
6,Paige Bueckers
6,Cotie McMahon
5,Natasha Cloud
5,Sabrina Ionescu


## The "David vs Goliath" Award

most shots blocked where the blocker is at least 5 inches shorter than the blockee

In [16]:
blocked_shots_w_player_heights = (
    pbp_curr_yr
    # missed shots
    .loc[pbp_curr_yr["shotResult"] == "Missed"]
    # join with block events
    .merge(
        pbp_curr_yr.loc[pbp_curr_yr["description"].str.contains("BLOCK", na=False)],
        on=["gameId", "clock", "period"],
        how="inner",
        suffixes=(".x", ".y")
    )
    # height of the shooter (blocked player)
    .merge(
        player_totals_w_gp_percentages[["PLAYER_ID", "PLAYER_HEIGHT_INCHES"]]
        .rename(columns={
            "PLAYER_ID": "personId.x",
            "PLAYER_HEIGHT_INCHES": "blocked_player_height"
        }),
        on="personId.x",
        how="left"
    )
    # height of the blocker
    .merge(
        player_totals_w_gp_percentages[["PLAYER_ID", "PLAYER_HEIGHT_INCHES"]]
        .rename(columns={
            "PLAYER_ID": "personId.y",
            "PLAYER_HEIGHT_INCHES": "blocking_player_height"
        }),
        on="personId.y",
        how="left"
    )
    .assign(
        height_diff=lambda d:
        d.blocked_player_height - d.blocking_player_height
    )
)

In [17]:
gt.GT(
        blocked_shots_w_player_heights
        .loc[lambda d: d["height_diff"] >= 5]
        .groupby(["personId.y", "playerNameI.y"], as_index=False)
        .size()
        .rename(columns={"size": "count"})
        .merge(
            player_totals_w_gp_percentages[["PLAYER_ID", "PLAYER_NAME"]]
            .rename(columns={
                "PLAYER_ID": "personId.y",
                "PLAYER_NAME": "player_name"
            }),
            on="personId.y",
            how="left"
        )
        .pipe(lambda d: d.assign(player_name=d.pop("player_name")))
        .drop(columns=["playerNameI.y", "personId.y"])
        .nlargest(10, "count", keep="all")
    )

count,player_name
20,Nia Brodie
20,Nia Brodie
20,Nia Coffey
20,Nia Coffey
8,Skylar Diggins
8,Courtney Williams
8,Julie Allemand
8,Flau'jae Johnson
6,Jordin Canada
5,Azzi Fudd


## The "Call Game" Award

most game winning points (defined as the first points that eclipsed the losing team's total) (credit to Necessary_Career_253 for the idea & Clownp3nis for the presenter)

In [18]:
game_winning_shots = (
    pbp_curr_yr
    .assign(
        vis_final=lambda d: d.groupby("gameId")["scoreAway"].transform("max"),
        home_final=lambda d: d.groupby("gameId")["scoreHome"].transform("max")
    )
    .assign(
        winner=lambda d: d["home_final"].gt(d["vis_final"]).map(
            {True: "home", False: "visitor"}
        ),
        loser_points=lambda d: d["vis_final"].where(
            d["home_final"] > d["vis_final"], d["home_final"]
        )
    )
    .loc[
        lambda d:
        ((d["winner"] == "home") & (d["scoreHome"] > d["loser_points"])) |
        ((d["winner"] == "visitor") & (d["scoreAway"] > d["loser_points"]))
    ]
    .sort_values("actionId")
    .groupby("gameId", as_index=False)
    .first()
    .assign(
        shotValue=lambda d: d["shotValue"].where(
            d["actionType"] != "Free Throw", 1
        )
    )
)

game_win_shot_summary = (
    game_winning_shots
    .groupby(["personId", "playerNameI"], as_index=False)
    .agg(
        game_winning_3=("shotValue", lambda s: (s == 3).sum()),
        game_winning_2=("shotValue", lambda s: (s == 2).sum()),
        game_winning_ft=("shotValue", lambda s: (s == 1).sum()),
        game_winning_points=("shotValue", "sum")
    )
    .merge(
        player_totals_w_gp_percentages[["PLAYER_ID", "PLAYER_NAME"]]
        .rename(columns={
            "PLAYER_ID": "personId",
            "PLAYER_NAME": "player_name"
        }),
        on="personId",
        how="left"
    )
    .pipe(lambda d: d.assign(player_name=d.pop("player_name")))
    .drop(columns="playerNameI")
)

In [19]:
gt.GT(
        game_win_shot_summary
        .nlargest(5, "game_winning_points", keep="all")
        .drop(columns="personId")
    ).cols_label(
        game_winning_3="GW3",
        game_winning_2="GW2",
        game_winning_ft="GWFT",
        game_winning_points="GWPTS"
    )

GW3,GW2,GWFT,GWPTS,player_name
0,10,1,21,Natasha Howard
2,4,4,18,A'ja Wilson
4,1,3,17,Janelle Salaun
4,1,3,17,Paige Bueckers
2,4,0,14,Chelsea Gray


## The Bowling Ball Award (sponsored by Pete Weber)*

most charges committed (credit to Kdog122025 for the idea)

In [20]:
violations = (
    pbp_curr_yr
    .loc[
        pbp_curr_yr["description"].str.contains(
            "Charge|Kick|Traveling|3 Second",
            na=False
        )
    ]
    .merge(
        player_totals_w_gp_percentages[["PLAYER_ID", "PLAYER_NAME"]]
        .rename(columns={
            "PLAYER_ID": "personId",
            "PLAYER_NAME": "player_name"
        }),
        on="personId",
        how="left"
    )
    .pipe(lambda d: d.assign(player_name=d.pop("player_name")))
    .drop(columns="playerNameI")
)

In [21]:
gt.GT(
    violations
    .loc[lambda d: d["description"].str.contains("Charge", na=False)]
    .groupby("player_name", as_index=False)
    .size()
    .rename(columns={"size": "count"})
    .nlargest(6, "count", keep="all")
)

player_name,count
Olivia Nelson-Ododa,9
Angel Reese,6
Nia Brodie,6
Nia Coffey,6
Aliyah Boston,5
Breanna Stewart,4
Cameron Brink,4
Dearica Hamby,4
Emily Engstler,4
Madina Okot,4


## "The Thing about Arsenal Is They Always Try to Walk It In" Award (presented by NWSL Commissioner Jessica Berman)*

most kicked ball violations

In [22]:
gt.GT(
    violations
    .loc[lambda d: d["description"].str.contains("Kick", na=False)]
    .groupby("player_name", as_index=False)
    .size()
    .rename(columns={"size": "count"})
    .nlargest(5, "count", keep="all")
)

player_name,count
Breanna Stewart,11
Natisha Hiedeman,9
Alyssa Thomas,8
Chelsea Gray,8
DeWanna Bonner,8
Kahleah Copper,8


## The "Pack Your Bags" Award (Sponsored by Delta, the global airline partner of the WNBA)

Player with the most traveling calls

In [23]:
gt.GT(
    violations
    .loc[lambda d: d["description"].str.contains("Traveling", na=False)]
    .groupby("player_name", as_index=False)
    .size()
    .rename(columns={"size": "count"})
    .nlargest(5, "count", keep="all")
)

player_name,count
Angel Reese,20
Madina Okot,12
Pauline Astier,10
Cecilia Zandalasini,8
Natasha Howard,8


## The "Holy Hand Grenade of Antioch" Award (presented by Sesame Street's Count von Count)

Player with the most illegal defense/defensive 3-second calls (credit to PsychoM & MrBuckBuck for the idea, asetniop for the name)

In [24]:
gt.GT(
    violations
    .loc[lambda d: d["description"].str.contains("3 Second", na=False)]
    .groupby("player_name", as_index=False)
    .size()
    .rename(columns={"size": "count"})
    .nlargest(5, "count", keep="all")
)

player_name,count
Lauren Betts,7
Angel Reese,6
Madina Okot,6
Aliyah Boston,4
Awa Fam,3
Dearica Hamby,3
Jonquel Jones,3
Kiki Iriafen,3


## The "Time is An Illusion" Award (sponsored by Salvador Dali)

team with the most 24-second shotclock, 8-second backcourt and 5-second inbound violations (credit to Necessary_Career_253 for the idea)

In [25]:
team_violations = (
    pbp_curr_yr
    .loc[
        (pbp_curr_yr["actionType"] == "Turnover") &
        (pbp_curr_yr["subType"].str.contains("Shot Clock|5 Second|8 Second", na=False))
    ]
    .assign(
        team_id=lambda d: (
            np.where(d['teamId'] == 0,d['personId'],d['teamId'])
        )
    )
)

team_violations_summary=(team_violations
    .groupby("team_id", as_index=False)
    .agg(
        shot_clock=("subType", lambda s: s.str.contains("Shot Clock", na=False).sum()),
        inbound=("subType", lambda s: s.str.contains("5 Second", na=False).sum()),
        halfcourt=("subType", lambda s: s.str.contains("8 Second", na=False).sum())
    )
    .assign(
        tot_violations=lambda d:
        d.shot_clock + d.inbound + d.halfcourt
    ).merge(
        right=team_info[['TeamID','TeamCity']],how='left',left_on='team_id',right_on='TeamID'
    )
    [['TeamCity','shot_clock','inbound','halfcourt','tot_violations']]
)

In [26]:
gt.GT(
    team_violations_summary
    .nlargest(5, "tot_violations", keep="all")
)

TeamCity,shot_clock,inbound,halfcourt,tot_violations
Golden State,46,1,2,49
New York,44,1,1,46
Dallas,42,2,1,45
Indiana,39,1,3,43
Portland,36,4,1,41


## The "I'll Have It to Go" Award (sponsored by DoorDash)

coach with lowest timeout utilization (credit to xfinityhomeboy, Ill_Ad3517 & s-sea (who also came up with the name))

In [27]:
num_timeouts_available = (
    pbp_curr_yr
    .loc[pbp_curr_yr["teamTricode"].notna()]
    .sort_values(["gameId", "period"])
    .groupby(["gameId", "period", "teamTricode"], as_index=False)
    .first()[["gameId", "period", "teamTricode"]]
    .rename(columns={"teamTricode": "team"})
    .groupby("team", as_index=False)
    .agg(
        games=("period", lambda s: (s <= 4).sum() / 4),
        overtimes=("period", lambda s: (s > 4).sum())
    )
    .assign(num_timeouts=lambda d: 6 * d.games + 3 * d.overtimes)
    .merge(
        team_info[["TeamName", "TEAM_ABBREVIATION"]],
        left_on="team",
        right_on="TEAM_ABBREVIATION",
        how="left"
    )
)

num_timeouts_used = (
    pbp_curr_yr
    .loc[
        pbp_curr_yr["description"].str.contains("Timeout", na=False) &
        ~pbp_curr_yr["description"].str.contains("Excess", na=False)
    ]
    [["gameId", "period", "description"]]
    .assign(
        team_name=lambda d: (
            d["description"]
            .str.split(" Timeout:", n=1)
            .str[0]
            .str.title()
        )
    )
    .groupby("team_name", as_index=False)
    .size()
    .rename(columns={"size": "num_timeouts_used"})
)

In [28]:
timeouts_used_percent = (
    num_timeouts_available
    .merge(
        num_timeouts_used,
        left_on="TeamName",
        right_on="team_name",
        how="left"
    )
    .assign(
        timeout_use_percent=lambda d:
        d["num_timeouts_used"] / d["num_timeouts"]
    )
    .sort_values("timeout_use_percent", ascending=False)
)

gt.GT(
    timeouts_used_percent
    [["team", "num_timeouts", "num_timeouts_used", "timeout_use_percent"]]
    .nsmallest(5, "timeout_use_percent", keep="all")
).fmt_percent(columns="timeout_use_percent")

team,num_timeouts,num_timeouts_used,timeout_use_percent
ATL,267.0,148,55.43%
MIN,264.0,160,60.61%
WAS,282.0,174,61.70%
LVA,276.0,172,62.32%
GSV,267.0,169,63.30%


## The "Hotheaded" Award (presented by Don Nelson)

most technicals plus ejections for non-players (credit to livejamie for the idea)

In [29]:
coach_ejection_techs = (
    pbp_curr_yr
    .loc[
        (pbp_curr_yr["teamId"] == 0) &
        (
            (pbp_curr_yr["actionType"] == "Ejection") |
            pbp_curr_yr["description"].str.contains(
                "T.FOUL|DOUBLE.TECHNICAL.FOUL",
                na=False
            )
        )
    ]
    .assign(
        coach=lambda d: (
            d["description"]
            .where(
                d["actionType"] == "Ejection",
                d["description"].str.split(" Foul:", n=1).str[0]
            )
            .where(
                d["actionType"] != "Ejection",
                d["description"].str.split(" Ejection:", n=1).str[0]
            )
            .str.title()
        )
    )
)

coach_ejection_tech_summary=(
    coach_ejection_techs
    .groupby("coach", as_index=False)
    .agg(
        ejections=("actionType", lambda s: (s == "Ejection").sum()),
        technicals=("actionType", lambda s: (s == "Foul").sum())
    )
    .assign(techs_plus_eject=lambda d: d.technicals + d.ejections)
)

In [30]:
gt.GT(coach_ejection_tech_summary.nlargest(5, "techs_plus_eject", keep="all"))

coach,ejections,technicals,techs_plus_eject
Chris Demarco,0,5,5
Becky Hammon,1,3,4
Lynne Roberts,0,4,4
Cheryl Reeve,0,3,3
Sydney Johnson,1,2,3


# ROUGH WORK